# Ch 24. デコーディング戦略 — 解答

> このノートブックには5問すべての再現可能な解答が含まれます。


## 問題 1

**問題**: 同じプロンプトからGreedy、Top-k=5、Top-p=0.9、Temperature=0.7で生成し比較せよ。

### 解法

Greedyは最大値を決定的に選ぶ。他は正規化した候補集合から標本化する。公平な比較では同じlogitとseedを使い、固有token数とlog確率を併記する。

以下のコードは乱数シードを固定した小規模CPU実験である。絶対時間や大規模な品質値は主張せず、比較すべき数学量と不変条件を検証する。


In [ ]:
import numpy as np
rng=np.random.default_rng(24); z=np.array([3.,2.2,1.5,1.,.4,-.2]);
def sample(z, mode):
 p=np.exp(z-z.max()); p/=p.sum()
 if mode=='greedy': return int(p.argmax())
 if mode=='topk': idx=np.argsort(p)[-5:]
 else:
  idx=np.argsort(p)[::-1]; idx=idx[np.cumsum(p[idx])-p[idx] < .9]
 q=p[idx]/p[idx].sum(); return int(rng.choice(idx,p=q))
out={m:[sample(z/.7 if m=='temp' else z, 'topp' if m=='temp' else m) for _ in range(200)] for m in ['greedy','topk','topp','temp']}
assert len(set(out['greedy']))==1 and all(0<=x<6 for v in out.values() for x in v)
{k:len(set(v)) for k,v in out.items()}


## 問題 2

**問題**: Temperature 0.3、0.7、1.0、1.5におけるエントロピーと生成多様性を比較せよ。

### 解法

温度$T$は$p_i(T)=\mathrm{softmax}(z_i/T)$を与える。低い$T$は分布を集中させentropyと多様性を概ね減らし、高い$T$は平坦化して増やす。

以下のコードは乱数シードを固定した小規模CPU実験である。絶対時間や大規模な品質値は主張せず、比較すべき数学量と不変条件を検証する。


In [ ]:
import numpy as np
z=np.array([3.,2.,1.,0.,-1.]); rows=[]
for T in [.3,.7,1.,1.5]:
 p=np.exp(z/T-(z/T).max()); p/=p.sum(); H=-np.sum(p*np.log(p)); rows.append((T,H,1/np.sum(p*p)))
assert all(rows[i][1] < rows[i+1][1] for i in range(3)); rows


## 問題 3

**問題**: Beam Searchのビーム幅1、4、8を比較せよ。

### 解法

beam searchは累積log確率上位$B$個のprefixを保持する。$B=1$はgreedyで、$B$増加により探索scoreは悪化しないが、長さbiasと多様性低下が起こり得る。

以下のコードは乱数シードを固定した小規模CPU実験である。絶対時間や大規模な品質値は主張せず、比較すべき数学量と不変条件を検証する。


In [ ]:
trans={():[(0,-.1),(1,-.2)],(0,):[(0,-2),(1,-.1)],(1,):[(0,-.2),(1,-.4)]}
def beam(B):
 beams=[((),0.)]
 for _ in range(2):
  cand=[]
  for seq,s in beams:
   opts=trans.get(seq,[(0,-.3),(1,-.3)])
   cand += [(seq+(t,),s+lp) for t,lp in opts]
  beams=sorted(cand,key=lambda x:x[1],reverse=True)[:B]
 return beams[0]
scores=[beam(B)[1] for B in [1,4,8]]
assert scores[0] <= scores[1] <= scores[2]; scores


## 問題 4

**問題**: Repetition Penalty 1.0、1.1、1.3の効果を実験せよ。

### 解法

共通乱数法を用い、各 seed で同一の一様乱数列をすべての penalty 条件に再利用する。64 seed の反復率の平均・標本標準偏差に加え、penalty 1.0 に対する paired 差の平均と標準誤差を報告する。標本平均の完全な単調性は要求せず、1-step の反復確率を独立に計算し、最も強い penalty の paired 効果を検証する。

以下は download 不要の小規模 CPU 実験で、実行時に計算した統計量を出力する。


In [ ]:
import numpy as np

base = np.array([2.5, 2.0, 1.0, 0.0])
penalties = (1.0, 1.1, 1.3)

def repeat_rate(penalty, uniforms, steps=400):
    seq = []
    for u in uniforms[:steps]:
        logits = base.copy()
        if seq:
            repeated = np.unique(seq)
            positive = logits[repeated] > 0
            logits[repeated[positive]] /= penalty
            logits[repeated[~positive]] *= penalty
        probs = np.exp(logits - logits.max())
        probs /= probs.sum()
        seq.append(int(np.searchsorted(np.cumsum(probs), u, side='right')))
    return np.mean(np.asarray(seq[1:]) == np.asarray(seq[:-1]))

seed_rates = {p: [] for p in penalties}
for seed in range(64):
    matched_uniforms = np.random.default_rng(seed).random(400)
    for penalty in penalties:
        seed_rates[penalty].append(repeat_rate(penalty, matched_uniforms))

summary = {
    p: {'mean': float(np.mean(v)), 'std': float(np.std(v, ddof=1))}
    for p, v in seed_rates.items()
}
paired = np.asarray(seed_rates[1.3]) - np.asarray(seed_rates[1.0])
summary['paired_1.3_minus_1.0'] = {
    'mean': float(paired.mean()),
    'standard_error': float(paired.std(ddof=1) / np.sqrt(len(paired))),
}

def probability_of_immediate_repeat(penalty, previous=0):
    logits = base.copy()
    logits[previous] /= penalty
    probs = np.exp(logits - logits.max())
    return float(probs[previous] / probs.sum())

reference = [probability_of_immediate_repeat(p) for p in penalties]
assert reference[0] > reference[1] > reference[2]
assert summary['paired_1.3_minus_1.0']['mean'] < 0
summary


## 問題 5

**問題**: Speculative Decodingが2〜3倍高速な理由を数学的に説明せよ。

### 解法

draft modelが一度に$k$ tokenを提案しtarget modelが並列検証する。平均承認率を$a$とするとtoken当たりtarget呼出費用は約$1/(1+ak)$、draft費用$c_d$込みのspeedupは$S\approx(1+ak)/(1+k c_d)$である。

以下のコードは乱数シードを固定した小規模CPU実験である。絶対時間や大規模な品質値は主張せず、比較すべき数学量と不変条件を検証する。


In [ ]:
def speedup(k,accept,draft_cost): return (1+k*accept)/(1+k*draft_cost)
vals=[speedup(4,a,.05) for a in [.5,.7,.9]]
assert vals[0] > 2 and vals[-1] < 4; vals
